In [ ]:
######## Zelle 1: Imports & IBM Login ########

import numpy as np
import pandas as pd

from qiskit_optimization import QuadraticProgram
from qiskit_ibm_runtime import QiskitRuntimeService, Sampler
from qiskit import transpile
from qiskit.circuit.library import QAOAAnsatz

# IBM Account laden
service = QiskitRuntimeService(channel="ibm_quantum")

print("✅ Service initialisiert")


In [4]:
######## Zelle 2: IBM Quantum Zugangsdaten ########

from qiskit_ibm_runtime import QiskitRuntimeService

# Deine Zugangsdaten hier eintragen
API_TOKEN = "hc_jCRxZWVqz8KZuZY58kmtbiC81I1tuGuKrU4a-VOGF"
CHANNEL = "ibm_quantum_platform"
INSTANCE = "crn:v1:bluemix:public:quantum-computing:us-east:a/71e0d8f4996f4919a7a1f5a17593eac9:817179d7-c733-47f0-89fa-64f2696e053c::"   # z. B. "ibm-q/open/main"
BACKEND = 'ibmtorino'
# Service initialisieren
service = QiskitRuntimeService(channel=CHANNEL, token=API_TOKEN, instance=INSTANCE)
print("IBM Quantum Service initialisiert ")


IBM Quantum Service initialisiert 


In [7]:
######## Zelle 3: Kleines Routing-Beispiel als QuadraticProgram ########
# 5 Knoten: 0 = Depot, 1..4 = Kunden
# einfache Zeitmatrix (euklidische Distanz * 10), inkl. "Priorität" als Reihenfolge-Constraint

import numpy as np
from qiskit_optimization import QuadraticProgram

np.random.seed(42)
coords = np.random.uniform(0, 100, size=(5, 2))
depot_idx = 0
prioritized_idx = 2     # Kunde 2 ist priorisiert (muss != 0 sein)
deadline_visit = 3      # spätestens als dritter Besuch (Position 1..4)
assert prioritized_idx != depot_idx, "prioritized_idx darf nicht das Depot sein."
deadline_visit = int(max(1, min(deadline_visit, coords.shape[0]-1)))  # clamp

# Zeitmatrix
def time_matrix_from_coords(xy):
    n = len(xy)
    m = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            if i != j:
                m[i, j] = np.linalg.norm(xy[i] - xy[j]) * 10.0
    return m

time_matrix = time_matrix_from_coords(coords)

def create_tsp_with_priority(time_matrix, prioritized_idx, deadline_visit):
    n = time_matrix.shape[0]
    qp = QuadraticProgram("tsp_priority")

    # Variablen x_{i,j} (binär): Fahrt i->j
    for i in range(n):
        for j in range(n):
            if i != j:
                qp.binary_var(f"x_{i}_{j}")

    # Reihenfolgevariablen u_i (MTZ), nur für Kunden (1..n-1)
    for i in range(1, n):
        qp.integer_var(1, n-1, f"u_{i}")  # <-- positional statt Keywords

    # Zielfunktion: Summe Fahrzeiten minimieren
    linear = {f"x_{i}_{j}": float(time_matrix[i, j]) for i in range(n) for j in range(n) if i != j}
    qp.minimize(linear=linear)

    # jeder Knoten wird einmal verlassen / erreicht
    for i in range(n):
        qp.linear_constraint(
            sense="==", rhs=1,
            linear={f"x_{i}_{j}": 1 for j in range(n) if i != j},
            name=f"leave_{i}"
        )
    for j in range(n):
        qp.linear_constraint(
            sense="==", rhs=1,
            linear={f"x_{i}_{j}": 1 for i in range(n) if i != j},
            name=f"enter_{j}"
        )

    # MTZ-Constraints gegen Subtouren (nur Kunden 1..n-1)
    for i in range(1, n):
        for j in range(1, n):
            if i != j:
                qp.linear_constraint(
                    sense="<=", rhs=n-2,
                    linear={f"u_{i}": 1, f"u_{j}": -1, f"x_{i}_{j}": n-1},
                    name=f"mtz_{i}_{j}"
                )

    # einfache Prioritäts-Deadline auf Reihenfolgevariable
    qp.linear_constraint(
        sense="<=", rhs=deadline_visit,
        linear={f"u_{prioritized_idx}": 1},
        name="priority_deadline"
    )
    return qp

qp = create_tsp_with_priority(time_matrix, prioritized_idx, deadline_visit)
print("✅ QuadraticProgram erstellt (TSP mit Priorität)")


✅ QuadraticProgram erstellt (TSP mit Priorität)


In [9]:
######## Zelle 4: QUBO -> Ising und QAOA-Ansatz (ohne qiskit_algorithms) ########

# QUBO
to_qubo = QuadraticProgramToQubo()
qubo = to_qubo.convert(qp)

# Ising-Operator (SparsePauliOp) + Offset
ising_op, offset = to_ising(qubo)

# QAOA Ansatz mit geringer Tiefe (p=1), um Hardware-Aufwand klein zu halten
reps = 1
qaoa = QAOAAnsatz(ising_op, reps=reps)
qc_param = qaoa.decompose()  # parametrischer Schaltkreis

# Heuristische Parameterbelegung (keine klassische Optimierung auf Hardware)
# Reihenfolge der Parameter in QAOAAnsatz: einfach alles auf 0.7 setzen
initial_params = [0.7] * len(qc_param.parameters)
qc = qc_param.assign_parameters(initial_params)

print(f"QAOA-Circuit erstellt. Parameteranzahl: {len(initial_params)} | Qubits: {qc.num_qubits}")


QAOA-Circuit erstellt. Parameteranzahl: 2 | Qubits: 66


In [22]:
######## Zelle 5a: Circuit vorbereiten ########

from qiskit import transpile

# Falls dein qc noch keine Messungen hat: alles messen
if not qc.clbits:
    qc.measure_all()
    print("ℹ️ Alle Qubits werden am Ende gemessen.")

# Backend auswählen
BACKEND = "ibm_brisbane"   # oder "ibm_torino"
backend_obj = service.backend(BACKEND)

# Circuit für das Backend transpilen
qc_transpiled = transpile(qc, backend=backend_obj)
print(f"✅ Circuit für {BACKEND} transpiliert.")
print("Qubits:", qc_transpiled.num_qubits, "| Clbits:", qc_transpiled.num_clbits)


ℹ️ Alle Qubits werden am Ende gemessen.
✅ Circuit für ibm_brisbane transpiliert.
Qubits: 127 | Clbits: 66


In [23]:
######## Zelle 5b: Job ausführen und Ergebnisse ########

from qiskit_ibm_runtime import Sampler

# Sampler mit Backend starten
sampler = Sampler(mode=backend_obj)

print(f"Sende Job an {BACKEND} ...")
job = sampler.run([qc_transpiled], shots=1024)
print("Job-ID:", job.job_id())

# Ergebnis abrufen
result = job.result()

# Quasi-Verteilungen holen (DataBin)
dist = result[0].data

# Wahrscheinlichkeiten
probs = dist.get_probabilities()
print("Wahrscheinlichkeiten:", probs)

# Klassische Counts (wie früher)
counts = dist.get_counts(shots=1024)
print("Counts:", counts)


Sende Job an ibm_brisbane ...
Job-ID: d2hinnlv7m7s73eaohs0


RuntimeInvalidStateError: 'Unable to retrieve result for job d2hinnlv7m7s73eaohs0. Job was cancelled.'

In [ ]:
######## Zelle 5c: Verfügbare Backends abfragen ########

backends = service.backends()   # alle für dich zugänglichen Backends
print("Gefundene Backends für deinen Account:\n")
for b in backends:
    status = "aktiv" if b.status().operational else "down"
    sim = " (Simulator)" if b.configuration().simulator else ""
    print(f"- {b.name}{sim} | {status}")


Gefundene Backends für deinen Account:

- ibm_brisbane | aktiv
- ibm_torino | aktiv


In [19]:
######## Zelle 6: Ergebnisse interpretieren (Bitstring -> Route-Heuristik) ########

# Sampler-Result (V2) hält Quasi-Verteilungen in result[0].data.meas.get_probabilities()
# Fallback: get_counts(), falls verfügbar
try:
    probs = result[0].data.meas.get_probabilities()
except Exception:
    counts = result[0].data.meas.get_counts()
    total = sum(counts.values())
    probs = {k: v/total for k, v in counts.items()}

# Bestes Bitstring
best_bitstring = max(probs, key=probs.get)
print("🔎 Bestes Bitstring:", best_bitstring)

# Heuristische Routenableitung:
# Für Demo mischen wir die Kundenreihenfolge deterministisch aus dem Bitstring-Seed
n = coords.shape[0]
rng_seed = int(best_bitstring, 2) % 10_000
rng = np.random.default_rng(rng_seed)
route = list(range(n))
rng.shuffle(route[1:])  # Depot (0) bleibt Start

# Prioritätskunde möglichst früh platzieren, wenn zu spät
if prioritized_idx in route[3:]:
    route.remove(prioritized_idx)
    route.insert(1, prioritized_idx)

# Route schließt zum Depot
route.append(0)

print("🚚 Abgeleitete Route:", route)

# (Optional) Speichern
pd.DataFrame({"Index": route}).to_csv("quantum_route.csv", index=False)
print("💾 Route in quantum_route.csv gespeichert")


AttributeError: 'DataBin' object has no attribute 'meas'